In [ ]:
# 读取videoChannels.txt文件，记录读取到哪一行。
import subprocess
from pathlib import Path

videoChannelsFilePath = Path(r"C:\02Programmer\02Proj\PyVSCode\ConfigPrivate\Rep001Tools002YTProjV4\videoChannels.txt")
def readFileLine(path: Path, line_number: int = 0, strip_newline: bool = True) -> list[str]:
    try:
        with path.open('r', encoding='utf-8') as f:
            if line_number <= 0:
                return [line.strip() for line in f if line.strip()]
            else:
                for current_line_num, line in enumerate(f, start=1):
                    if current_line_num == line_number:
                        return [line.rstrip('\n') if strip_newline else line]
    except FileNotFoundError:
        print(f"文件未找到: {path}")
    except Exception as e:
        print(f"读取文件时出错: {e}")
    return []

readFileLine(videoChannelsFilePath, 1)


In [ ]:
def wait_for_user_input():
    """程序暂停，等待用户输入'y'继续或'n'退出"""
    while True:
        user_input = input("输入 'y' 继续，输入 'n' 退出: ")
        if user_input.lower() == 'n':
            return False  # 用户选择退出，返回 False
        elif user_input.lower() == 'y':
            return True  # 用户选择继续，返回 True
        else:
            print("无效输入，请输入 'y' 或 'n'。")
            
# wait_for_user_input()

In [ ]:
# 从 Firefox 获取 cookies 并 格式化为 Netscape 格式（yt-dlp 支持的格式）
import time
import browser_cookie3

cookies_path = Path(r"C:\02Programmer\02Proj\PyVSCode\ConfigPrivate\Rep001Tools002YTProjV4\cookies2.txt")     # 用于身份验证的 cookies 路径

def reget_cookies(cookies_path: Path):
    # 从 Firefox 获取 cookies
    cookies = browser_cookie3.firefox(domain_name='youtube.com')

    # 格式化为 Netscape 格式（yt-dlp 支持的格式）
    cookies_txt_path = cookies_path

    with open(cookies_txt_path, 'w', encoding='utf-8') as f:
        f.write("# Netscape HTTP Cookie File\n")
        for cookie in cookies:
            domain = cookie.domain if cookie.domain.startswith('.') else '.' + cookie.domain
            path = cookie.path or '/'
            secure = "TRUE" if cookie.secure else "FALSE"
            expires = int(time.time()) + 3600 * 24 * 30  # 设置过期时间为 30 天
            f.write(f"{domain}\tTRUE\t{path}\t{secure}\t{expires}\t{cookie.name}\t{cookie.value}\n")
    print(f"Cookies 已保存到: {cookies_txt_path}")
    
# reget_cookies(cookies_path)


In [ ]:
example_url = "https://www.youtube.com//videos"
# 提取播放列表所有视频链接
def extract_video_urls(playlist_url: str) -> list[str]:
    cmd = [
        "yt-dlp", "--flat-playlist", "--print", "url",
        "--cookies", str(cookies_path),
        "--match-filter", "is_live = False",  # 排除直播与预告
        "--yes-playlist",   # 明确表示处理整个频道/列表
        playlist_url
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    urls = result.stdout.strip().splitlines()
    return [f"https://www.youtube.com/watch?v={u}" if "http" not in u else u for u in urls]

# 示例使用
extract_video_urls(example_url)

In [ ]:
# 🛠 进阶结构优化建议（非必须）
# 🧩 1. 将所有参数组合打包为模块字典，提升可维护性
# 你现在是扁平的 list + 拼接，如果后期加条件判断（例如配置文件中开关某个功能），会更麻烦。

# ✅ 推荐改写为：
# yt_dlp_options = {
#     "core": use_cookies + prefer_ffmpeg + video_format + merge_mp4,
#     "metadata": embed_metadata,
#     "filtering": view_count_filter + exclude_filter + exclude_live + skip_no_video,
#     "output": output_template + print_filepath + output_na_safe,
#     "archive": download_archive + no_overwrite,
#     "resilience": retry_on_failure + utf8_encoding + concurrent_fragments + show_progress + no_call_home,
#     "interval": sleep_between_downloads,
# }

# def build_cmd(url: str) -> list[str]:
#     cmd = [yt_dlp_binary]
#     for section in yt_dlp_options.values():
#         cmd += section
#     cmd.append(url)
#     return cmd

# ✅ 好处：

# 可动态开关模块（如 if config['metadata']: cmd += yt_dlp_options["metadata"]）

# 支持未来从配置文件控制启用哪些功能

In [ ]:
# === 配置项 ===
output_root = Path(r"G:\YTDowmLoad\video")  # 下载目标路径

# === yt-dlp 执行器 ===
yt_dlp_binary = "yt-dlp"  # 可替换为完整路径，如 r"C:\yt-dlp\yt-dlp.exe"

# === yt-dlp 参数组合（按模块整理 + 注释一行）===

# 登录与核心设置
use_cookies = ["--cookies", str(cookies_path)]  # 使用 cookies 登录/解锁区域限制
prefer_ffmpeg = ["--prefer-ffmpeg"]  # 使用 ffmpeg 而不是原生合并器
video_format = ["-f", "bv*+ba/b"]  # 下载最佳视频+音频流（否则回退到综合流）
merge_mp4 = ["--merge-output-format", "mp4"]  # 合并输出为 mp4 格式

# 元数据与字幕（可选开启）
embed_metadata = ["--embed-metadata"]  # 将标题、作者等元数据写入文件
# embed_thumbnail = ["--embed-thumbnail"]  # 将缩略图作为封面嵌入视频
# write_info_json = ["--write-info-json"]  # 保存 info.json 元数据文件
# add_metadata = ["--add-metadata"]  # 添加上传者、标签等元数据到文件
# embed_subtitles = ["--embed-subs"]  # 嵌入字幕（可选）
# write_subtitles = ["--write-subs"]  # 下载字幕（可选）

# 下载过滤与控制
view_count_filter = ["--match-filter", "view_count > 100000"]  # 只下载播放量 >10万 的视频
exclude_filter = ["--reject-title", "Live"]  # 排除标题包含 "Live" 的视频
exclude_live = ["--match-filter", "is_live = False"]  # 排除正在直播或预定直播中的视频。常用于频道视频列表、播放列表中，防止 yt-dlp 下载到未开放的视频流。这条规则会跳过：正在直播的视频（即“直播中”）;预告但尚未开始的直播
download_archive = ["--download-archive", str(output_root / "videoDownloaded.txt")]  # 已下载记录
no_overwrite = ["--no-post-overwrites"]  # 避免重新处理已存在视频

# 输出路径与命名
output_template = ["--output", str(output_root / "%(channel_handle)s/%(upload_date)s-%(title)s.80s.%(ext)s")]  # 输出结构
print_filepath = ["--print", "after_move:filepath"]  # 下载完成后打印文件路径
output_na_safe = ["--output-na-placeholder", "_"]  # 替代无元数据的字段，推荐; 输出命名容错：避免标题缺失导致命名出错
skip_no_video = ["--match-filter", "duration > 10"]  # 跳过极短非视频内容

# 容错与下载优化
retry_on_failure = ["--retries", "5", "--fragment-retries", "5"]  # 下载失败时重试
utf8_encoding = ["--encoding", "utf-8"]  # 防止文件名乱码
concurrent_fragments = ["--concurrent-fragments", "3"]  # 并发片段数
show_progress = ["--progress"]  # 显示下载进度条
# verbose_log = ["--verbose"]  # 输出详细日志（调试期可开）
no_call_home = ["--no-call-home"]  # 不发送使用统计信息

# 下载间隔控制（模拟人类行为）
sleep_between_downloads = ["--sleep-interval", "3", "--max-sleep-interval", "7"]  # 每个视频之间间隔 3~7 秒

# === 构造下载命令 ===
def build_cmd(url: str) -> list[str]:
    return (
        [yt_dlp_binary] +
        use_cookies +
        prefer_ffmpeg +
        video_format +
        merge_mp4 +

        embed_metadata +
        # embed_thumbnail +
        # write_info_json +
        # add_metadata +
        # embed_subtitles + write_subtitles +

        view_count_filter +
        exclude_filter +
        exclude_live +
        download_archive +
        no_overwrite +

        output_template +
        print_filepath +
        output_na_safe +
        skip_no_video +

        retry_on_failure +
        utf8_encoding +
        concurrent_fragments +
        show_progress +
        no_call_home +
        sleep_between_downloads +
        [url]
    )

# 示例使用
cmd = build_cmd(example_url)

# 然后可以用 subprocess 执行:
# subprocess.run(cmd, check=True)
# try:
#     result = subprocess.run(cmd, check=True, capture_output=True, text=True)
# except subprocess.CalledProcessError as e:
#     print(f"[stderr] {e.stderr}")

In [ ]:
line_num = 1
already_reget_cookies = False  # 是否需要重新获取 cookies

while True:
    # 读取下一行
    line = readFileLine(videoChannelsFilePath, line_number=line_num)
    if not line:
        print("已经是文件末尾,取不出来链接了。")
        # 如果没有更多行，暂停并等待用户输入，让用户增加更多行
        if not wait_for_user_input():  # 如果用户选择退出，停止程序
            print("没有更多内容，程序终止。")
            break
        # 如果用户输入'y'，继续
        print("继续处理下一行...")
        continue  # 继续处理下一行

    # 处理读取到的行
    playlistUrl = line[0]  # 假设每行只有一个 URL
    extractUrls = extract_video_urls(playlistUrl)
    for url in extractUrls:
        cmd = build_cmd(url)
        try:
            result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        except subprocess.CalledProcessError as e:
            print(f"[stderr] {e.stderr}")
            continue
        
    print(f"下载完第 {line_num} 行。")
    line_num += 1  # 更新行号


| 模板字段                 | 示例值                                               | 来源                   |
| -------------------- | ------------------------------------------------- | -------------------- |
| `%(uploader)s`       | Ggotbbang Official                                | 视频页面显示的上传者名称         |
| `%(uploader_id)s`    | UCxxxx...                                         | 渠道唯一 ID，格式 `UC` 开头   |
| `%(channel)s`        | Ggotbbang Official                                | 与 `uploader` 通常一致    |
| `%(channel_id)s`     | UCxxxx...                                         | 与 `uploader_id` 通常一致 |
| `%(channel_url)s`    | [https://youtube.com/@](https://youtube.com/@)... | 完整主页链接（不推荐用于文件夹名）    |
| `%(channel_handle)s` | @ggotbbang\_official                              | 主页链接里的用户名 ✅          |


| 字段                | 含义                   |
| ----------------- | -------------------- |
| `upload_date`     | 上传日期，格式为 `YYYYMMDD`  |
| `release_date`    | 发布时间（如果是音乐类/电影类），也可选 |
| `title`           | 视频标题                 |
| `uploader`        | 上传者名                 |
| `id`              | 视频 ID                |
| `ext`             | 文件扩展名                |
| `duration_string` | 时长（格式如 3:45）         |
| `view_count`      | 播放量                  |
| `like_count`      | 点赞量                  |

